# Example 2: Early Down Passing Efficiency

### Hypothesis
Teams that pass more often on early downs (1st and 2nd) are more efficient offensively.

EPA is expected points added.

### Tasks
* Filter to 1st and 2nd down
* Filter to neutral game script
    - This will be defined as a point differential of < 8 points to avoid skewdness in late game comebacks
* Calculate early donw pass rate
    - Just use the mean for this because pass play = 1
* Look for correlations
* Linear regression

In [31]:
# Imports
import nfl_data_py as nfl
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import numpy as np
import statsmodels.api as sm


In [32]:
# Get play-by-play dataframe
pbp = nfl.import_pbp_data([2025])
pbp.head()

2025 done.
Downcasting floats.


,play_id,game_id,old_game_id_x,home_team,away_team,season_type,week,posteam,posteam_type,defteam,...,was_pressure,route,defense_man_zone_type,defense_coverage_type,offense_names,defense_names,offense_positions,defense_positions,offense_numbers,defense_numbers
0,1.0,2025_01_ARI_NO,2025090705,NO,ARI,REG,1,None,None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,40.0,2025_01_ARI_NO,2025090705,NO,ARI,REG,1,ARI,away,NO,...,0.0,,,None,Denzel Burke;Joey Blount;Dadrion Taylor-Demers...,Rejzohn Wright;Quincy Riley;Ugo Amadi;Julian B...,CB;FS;FS;ILB;ILB;RB;S;TE;TE;TE;WR,CB;CB;FS;FS;ILB;ILB;K;MLB;OLB;RB;SS,29;32;42;44;50;31;36;84;81;87;4,25;29;0;23;53;44;19;28;40;13;33
2,63.0,2025_01_ARI_NO,2025090705,NO,ARI,REG,1,ARI,away,NO,...,0.0,,,None,Hjalte Froholdt;Evan Brown;Isaiah Adams;Kyler ...,Isaac Yiadom;Kool-Aid McKinstry;Nathan Shepher...,C;G;G;QB;RB;T;T;TE;TE;TE;WR,CB;CB;DT;DT;FS;ILB;MLB;NT;OLB;OLB;SS,72;63;74;1;6;73;70;85;84;87;18,27;4;93;90;23;20;56;92;94;96;21
3,85.0,2025_01_ARI_NO,2025090705,NO,ARI,REG,1,ARI,away,NO,...,0.0,QUICK OUT,ZONE_COVERAGE,COVER_2,Hjalte Froholdt;Evan Brown;Isaiah Adams;Kyler ...,Isaac Yiadom;Kool-Aid McKinstry;Nathan Shepher...,C;G;G;QB;RB;T;T;TE;TE;WR;WR,CB;CB;DT;DT;FS;ILB;MLB;NT;OLB;OLB;SS,72;63;74;1;6;73;70;85;87;14;18,27;4;93;90;23;20;56;92;94;96;21
4,115.0,2025_01_ARI_NO,2025090705,NO,ARI,REG,1,ARI,away,NO,...,1.0,,ZONE_COVERAGE,COVER_6,Hjalte Froholdt;Evan Brown;Isaiah Adams;Kyler ...,Isaac Yiadom;Kool-Aid McKinstry;Nathan Shepher...,C;G;G;QB;RB;T;T;TE;TE;WR;WR,CB;CB;DT;DT;FS;ILB;MLB;NT;OLB;OLB;SS,72;63;74;1;6;73;70;85;87;17;18,27;4;93;90;23;20;56;92;94;96;21


In [33]:
# Keep only the columns we need and make a copy of original df
pbp_small = pbp[[
    'down',
    'ydstogo',
    'epa',
    'play_type',
    'posteam',
    'yardline_100',
    'game_id',
    'season',
    'play_id',
    'score_differential',
    'pass',
    'defteam_score_post',
    'posteam_score_post'
]].copy()

pbp_small.head()

,down,ydstogo,epa,play_type,posteam,yardline_100,game_id,season,play_id,score_differential,pass,defteam_score_post,posteam_score_post
0,NaN,0.0,-0.000000,None,None,NaN,2025_01_ARI_NO,2025,1.0,NaN,0.0,NaN,NaN
1,NaN,0.0,-0.352700,kickoff,ARI,35.0,2025_01_ARI_NO,2025,40.0,0.0,0.0,0.0,0.0
2,1.0,10.0,-0.190052,run,ARI,78.0,2025_01_ARI_NO,2025,63.0,0.0,0.0,0.0,0.0
3,2.0,7.0,1.317340,pass,ARI,75.0,2025_01_ARI_NO,2025,85.0,0.0,1.0,0.0,0.0
4,1.0,10.0,-1.694360,pass,ARI,64.0,2025_01_ARI_NO,2025,115.0,0.0,1.0,0.0,0.0


In [34]:
# Filter dataframe to early down, less than 8 point differential, and pass/run plays
early_down = pbp_small[
    (pbp['down'].isin([1,2])) &
    (pbp['play_type'].isin(['pass', 'run'])) &
    (pbp['score_differential'].abs() <= 8)
].copy()
early_down.shape

(17700, 13)

In [35]:
# Calculate pass rate, epa_per_play, and total plays by team
team_rates = (
    early_down
    .groupby('posteam')
    .agg(
        pass_rate=('pass', 'mean'),
        epa_per_play=('epa', 'mean'),
        plays=('epa', 'count')
    )
    .reset_index()
)
team_rates.head()

,posteam,pass_rate,epa_per_play,plays
0,ARI,0.626638,-0.043008,458
1,ATL,0.503975,0.089923,629
2,BAL,0.458491,0.042228,530
3,BUF,0.520134,0.082064,596
4,CAR,0.505515,-0.066218,544


## Pass Rate vs EPA

In [36]:
# Scatter plot of pass rate vs EPA
fig = px.scatter(
    team_rates,
    x='pass_rate',
    y='epa_per_play',
    hover_data=['posteam'],
    trendline='ols',
    title='Early Down Pass Rate vs. Offensive Efficiency',
    labels={
        'pass_rate': 'Early Down Pass Rate',
        'epa_per_play': 'Early Down Pass Efficiency'
    }
)
fig.show()

In [37]:
# Measure strength of relationship b/t early down pass rates and epa per play
team_rates['pass_rate'].corr(
    team_rates['epa_per_play']
)

0.08565249497504175

The trendline is basically flat, showing **no correlation**. The correlation coefficient of 0.09 also points to no correlation between early down passes and expected points added per play.

## Linear Regression

In [38]:
X = team_rates['pass_rate']
y = team_rates['epa_per_play']

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:           epa_per_play   R-squared:                       0.007
Model:                            OLS   Adj. R-squared:                 -0.026
Method:                 Least Squares   F-statistic:                    0.2217
Date:                Tue, 17 Mar 2026   Prob (F-statistic):              0.641
Time:                        13:00:26   Log-Likelihood:                 37.148
No. Observations:                  32   AIC:                            -70.30
Df Residuals:                      30   BIC:                            -67.36
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0680      0.176     -0.386      0.7

In [39]:
pass_epa = early_down[
    early_down['pass'] == 1
].groupby('posteam')['epa'].mean()

pass_epa

posteam
ARI    0.011874
ATL    0.190183
BAL    0.122859
BUF    0.166455
CAR   -0.056376
CHI    0.133608
CIN   -0.051770
CLE   -0.111072
DAL    0.232709
DEN    0.063709
DET    0.178072
GB     0.260625
HOU    0.037484
IND    0.090112
JAX    0.085907
KC     0.079235
LA     0.259690
LAC    0.082509
LV    -0.101626
MIA    0.092962
MIN   -0.000707
NE     0.215620
NO    -0.009833
NYG    0.071062
NYJ   -0.104674
PHI    0.060983
PIT    0.056932
SEA    0.189144
SF     0.138210
TB     0.130359
TEN   -0.036612
WAS    0.129600
Name: epa, dtype: float32

In [40]:
pass_epa = early_down[
    early_down['pass'] == 0
].groupby('posteam')['epa'].mean()

pass_epa

posteam
ARI   -0.135119
ATL   -0.011944
BAL   -0.026041
BUF   -0.009409
CAR   -0.076280
CHI    0.016460
CIN    0.017291
CLE   -0.139697
DAL   -0.001238
DEN   -0.047566
DET    0.017275
GB    -0.071828
HOU   -0.162096
IND    0.004148
JAX    0.024033
KC    -0.082696
LA     0.082407
LAC   -0.143610
LV    -0.179172
MIA    0.056216
MIN   -0.077885
NE    -0.086488
NO    -0.126524
NYG   -0.070744
NYJ   -0.184865
PHI   -0.092963
PIT   -0.043253
SEA   -0.118175
SF    -0.085175
TB    -0.055611
TEN   -0.093715
WAS   -0.090460
Name: epa, dtype: float32

## Pass Rate vs. Win Rate

In [41]:
# one row per game showing final score
games = pbp_small.groupby(
    ['game_id', 'posteam']
).agg(
    points_scored=('posteam_score_post', 'max'),
    points_allowed=('defteam_score_post', 'max')
).reset_index()

games.head()

,game_id,posteam,points_scored,points_allowed
0,2025_01_ARI_NO,ARI,20.0,13.0
1,2025_01_ARI_NO,NO,13.0,20.0
2,2025_01_BAL_BUF,BAL,40.0,38.0
3,2025_01_BAL_BUF,BUF,41.0,40.0
4,2025_01_CAR_JAX,CAR,10.0,26.0


In [42]:
# Determin wins
games['win'] = (
    games['points_scored'] > games['points_allowed']
).astype(int)

games.head()

,game_id,posteam,points_scored,points_allowed,win
0,2025_01_ARI_NO,ARI,20.0,13.0,1
1,2025_01_ARI_NO,NO,13.0,20.0,0
2,2025_01_BAL_BUF,BAL,40.0,38.0,1
3,2025_01_BAL_BUF,BUF,41.0,40.0,1
4,2025_01_CAR_JAX,CAR,10.0,26.0,0


In [43]:
# Calculate team win rate
team_wins = games.groupby('posteam').agg(
    win_rate=('win', 'mean'),
    games=('win', 'count')
).reset_index()

team_wins

,posteam,win_rate,games
0,ARI,0.235294,17
1,ATL,0.470588,17
2,BAL,0.529412,17
3,BUF,0.684211,19
4,CAR,0.444444,18
5,CHI,0.631579,19
6,CIN,0.352941,17
7,CLE,0.294118,17
8,DAL,0.411765,17
9,DEN,0.842105,19


In [44]:
# Merge with Pass Rate Data
study_df = team_rates.merge(
    team_wins,
    on='posteam'
)

study_df.head()

,posteam,pass_rate,epa_per_play,plays,win_rate,games
0,ARI,0.626638,-0.043008,458,0.235294,17
1,ATL,0.503975,0.089923,629,0.470588,17
2,BAL,0.458491,0.042228,530,0.529412,17
3,BUF,0.520134,0.082064,596,0.684211,19
4,CAR,0.505515,-0.066218,544,0.444444,18


In [45]:
# Plot early down pass rate vs win rate
fig = px.scatter(
    study_df,
    x='pass_rate',
    y='win_rate',
    hover_data=['posteam'],
    trendline='ols',
    title='Early Down Pass Rate vs. Win Rate',
    labels={
        'pass_rate': 'Early Down Pass Rate',
        'win_rate': 'Win Rate'
    }
)

fig.show()

In [46]:
# Calculate Correlation
study_df['pass_rate'].corr(
    study_df['win_rate']
)

0.16203708803102856

This shows a weak positvie relationship between early down pass rate and win rate.